# EN3160 Assignment 2 — guided notebook
Work through this notebook **one cell at a time**, from top to bottom. Click a code cell and press **Shift+Enter**. Read the explanation above each cell. Do not use **Run All** on your first attempt because Question 3 needs mouse clicks and your own images.

Fill these in before writing your report:
- Name: ...
- Index number: ...
- GitHub profile: ...

The submitted report must be **4 pages or fewer**, named `your_index_a02.pdf`. It must contain your name/index, GitHub profile link, important code, results, comparisons and discussion. This notebook supports your work; it is not the submitted PDF.

## 0. Windows setup
1. Create a folder called `EN3160_A02`, and open it using **File → Open Folder** in VS Code.
2. Put this notebook inside it. Extract the supplied `a2images.zip` into the same folder. There should be a folder called `a2images`, with `pcbs` inside it. If extraction adds another outer `a2images` folder, use the inner folder containing the sunflower JPEG and `pcbs`.
3. Install the Microsoft **Python** and **Jupyter** extensions in VS Code (Ctrl+Shift+X).
4. Open **Terminal → New Terminal**. Run these **terminal commands**, one line at a time:

```powershell
py -m venv .venv
.\.venv\Scripts\python.exe -m pip install --upgrade pip
.\.venv\Scripts\python.exe -m pip install "opencv-python>=4.8,<5" numpy scipy matplotlib ipykernel ipympl
```

If `py` is unavailable but Python is installed, use `python -m venv .venv` for the first command. We invoke the virtual environment's Python directly, so PowerShell activation is unnecessary.
5. Open this notebook. At the top right, click **Select Kernel → Python Environments → .venv**. If it is missing, choose the interpreter at `EN3160_A02\.venv\Scripts\python.exe`.
6. Run the following cell. The output should include `Environment ready`.

In [ ]:
from pathlib import Path
import sys, json, csv
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_laplace, maximum_filter
from scipy.optimize import minimize

print("Python:", sys.executable)
print("OpenCV:", cv.__version__)
assert hasattr(cv, "SIFT_create"), "Install opencv-python in this notebook's environment."
print("Environment ready")

### Check the folders
`DATA` must point to the folder containing the sunflower JPEG. Change only `DATA` if your extracted folder has a different name. All results will be saved automatically inside `outputs`.

If you get a file error, read the printed current folder. Open the project folder in VS Code and keep the notebook at its top level.

In [ ]:
DATA = Path("a2images")
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
print("Current folder:", Path.cwd())
required = [DATA / "the_berry_farms_sunflower_field.jpeg", DATA / "pcbs" / "AECC1.jpg"]
for path in required:
    print(path, "OK" if path.exists() else "MISSING")
assert all(p.exists() for p in required), "Correct DATA before continuing."

def read_bgr(path):
    im = cv.imread(str(path))
    if im is None:
        raise FileNotFoundError(f"Could not read {path}")
    return im

def show_bgr(im, title="", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
    return ax

## 1. Sunflower blob detection
A blob is an approximately circular image region. Here we detect the **dark circular flower centres**. These circles describe the centres, rather than the complete petal boundaries.

We use a scale-normalized Laplacian of Gaussian:
\[
R(x,y,\sigma)=\sigma^2\nabla^2(G_\sigma*I)(x,y).
\]
For dark blobs, the positive response is useful. A local maximum in **x, y and scale** gives a detected centre and scale. We estimate its radius by \(r\approx\sqrt{2}\sigma\). This is an ideal circular-blob approximation.

### 1.1 Read the image at the required reduced resolution
OpenCV uses BGR colour order. Convert to RGB only when displaying with Matplotlib.

In [ ]:
sunflower = cv.imread(str(DATA / "the_berry_farms_sunflower_field.jpeg"),
                     cv.IMREAD_REDUCED_COLOR_4)
assert sunflower is not None
gray = cv.cvtColor(sunflower, cv.COLOR_BGR2GRAY).astype(np.float64) / 255.0
print("Reduced image shape:", sunflower.shape)
show_bgr(sunflower, "Reduced sunflower image")
plt.show()

### 1.2 Build the scale space and find extrema
The sigma range is **1.5–24 pixels**, with 32 logarithmically spaced scales, in the reduced image. We exclude the two end scales because a scale maximum there has not been bracketed on both sides.

`THRESHOLD` removes weak responses. The radius is in reduced-image pixels. The next cell also filters candidates using a yellow ring around the dark centre and a field region below 52% of the image height. These are image-specific heuristics; **the blob detection itself is LoG with scale-space extrema**, not a circle Hough transform. Explain the heuristics in your report.

In [ ]:
SIGMAS = np.geomspace(1.5, 24.0, 32)
THRESHOLD = 0.18
FIELD_START = 0.52
MIN_YELLOW_FRACTION = 0.50

responses = np.stack([s**2 * gaussian_laplace(gray, sigma=s)
                      for s in SIGMAS], axis=-1)
maxima = (responses == maximum_filter(responses, size=(3, 3, 3)))
maxima &= responses > THRESHOLD
maxima[:, :, 0] = False
maxima[:, :, -1] = False
ys, xs, scales = np.where(maxima)
candidates = sorted(zip(xs, ys, scales),
                    key=lambda p: responses[p[1], p[0], p[2]], reverse=True)

hsv = cv.cvtColor(sunflower, cv.COLOR_BGR2HSV)
yellow = cv.inRange(hsv, np.array([15, 85, 70]), np.array([40, 255, 255])) > 0
height, width = gray.shape
blobs = []  # x, y, radius, sigma, response
for x, y, k in candidates:
    sigma = SIGMAS[k]
    radius = np.sqrt(2) * sigma
    if y < FIELD_START * height:
        continue
    if x-radius < 0 or y-radius < 0 or x+radius >= width or y+radius >= height:
        continue  # exclude circles cut by the image boundary
    left, right = max(0, int(x-2*radius)), min(width, int(x+2*radius)+1)
    top, bottom = max(0, int(y-2*radius)), min(height, int(y+2*radius)+1)
    yy, xx = np.mgrid[top:bottom, left:right]
    distance = np.hypot(xx-x, yy-y)
    ring = (distance >= radius) & (distance <= 2*radius)
    if not ring.any() or yellow[top:bottom, left:right][ring].mean() < MIN_YELLOW_FRACTION:
        continue
    # Avoid multiple detections of the same centre.
    if any(np.hypot(x-b[0], y-b[1]) < 0.8*(radius+b[2]) for b in blobs):
        continue
    blobs.append((float(x), float(y), float(radius), float(sigma),
                  float(responses[y, x, k])))

blobs.sort(key=lambda b: b[2], reverse=True)
print(f"Sigma range: {SIGMAS[0]:.1f} to {SIGMAS[-1]:.1f} reduced-image pixels")
print("Detected circles:", len(blobs))
print("Five largest: x, y, radius, sigma, response")
for blob in blobs[:5]:
    print(tuple(round(v, 3) for v in blob))
with open(OUT / "q1_largest_circles.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["x_reduced", "y_reduced", "radius_reduced", "sigma_reduced", "response"])
    writer.writerows(blobs[:5])

### 1.3 Draw and save the circles
Coordinates use **x to the right, y downward, origin at the top left**. The five largest detections are numbered. Inspect them: a detected circle is not automatically a correct flower. If needed, adjust the threshold or yellow-ring fraction and rerun 1.2 and 1.3. Keep an honest note about missed flowers and false detections.

To express a measurement in the original 1440 × 1440 image, multiply the reduced-image coordinates and radius by 4; report which resolution you used.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
show_bgr(sunflower, "LoG sunflower-centre detections", ax)
for i, (x, y, radius, sigma, strength) in enumerate(blobs):
    ax.add_patch(plt.Circle((x, y), radius, fill=False,
                           color="lime" if i < 5 else "red", linewidth=0.8))
    if i < 5:
        ax.text(x, y, str(i+1), color="white", fontsize=9)
fig.tight_layout()
fig.savefig(OUT / "q1_sunflower_circles.png", dpi=200, bbox_inches="tight")
plt.show()

**For the report:** include the overlay, sigma range, response threshold, filtering rules and the largest-circle table. Mention that the method estimates circular dark centres; perspective and noncircular shapes limit its accuracy. Try one other threshold, for example 0.14, and compare the visible results.

**Commit checkpoint:** save the notebook and commit your Q1 work using VS Code Source Control (instructions at the end).

## 2. Fit a line, then a circle, using your own RANSAC
First generate the supplied dataset. We set a seed so reruns are reproducible. The true labels are used only for the initial illustration and evaluation, **never for selecting RANSAC inliers**.

There are 50 circle points and 50 line points. The circle has centre (2, 3), radius 10, radial noise standard deviation 0.625. The line is y = −x + 2 with vertical noise standard deviation 1.

In [ ]:
np.random.seed(0)
N = 100
half_n = N // 2
r_gt = 10.0
x0_gt, y0_gt = 2.0, 3.0
t = np.random.uniform(0, 2*np.pi, half_n)
n = (r_gt/16) * np.random.randn(half_n)
x = x0_gt + (r_gt+n)*np.cos(t)
y = y0_gt + (r_gt+n)*np.sin(t)
X_circ = np.column_stack((x, y))
x = np.linspace(-12, 12, half_n)
y = -x + 2 + np.random.randn(half_n)
X_line = np.column_stack((x, y))
X = np.vstack((X_circ, X_line))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_circ[:, 0], X_circ[:, 1], s=20, label="Generated circle points")
ax.scatter(X_line[:, 0], X_line[:, 1], s=20, label="Generated line points")
ax.set_aspect("equal"); ax.grid(alpha=0.25); ax.legend()
ax.set_title("Noisy point set")
plt.show()

### 2.1 Line model and perpendicular error
Use \(ax+by=d\), with \(a^2+b^2=1\) and \(d\geq0\). The distance from a point is then \(|ax+by-d|\). Two distinct points give a sample line.

The final line is fitted to its consensus using total least squares (TLS). The eigenvector with the smallest covariance eigenvalue is the unit normal, so the constraint is automatically satisfied.

In [ ]:
def canonical_line(normal, d):
    length = np.linalg.norm(normal)
    normal = np.asarray(normal, dtype=float) / length
    d = float(d) / length
    if d < 0:
        normal, d = -normal, -d
    return np.array([normal[0], normal[1], d])

def line_from_two(sample):
    direction = sample[1] - sample[0]
    if np.linalg.norm(direction) < 1e-10:
        return None
    normal = np.array([-direction[1], direction[0]])
    return canonical_line(normal, normal @ sample[0])

def line_error(points, model):
    return np.abs(points @ model[:2] - model[2])

def tls_line(points):
    centre = points.mean(axis=0)
    shifted = points - centre
    _, vectors = np.linalg.eigh(shifted.T @ shifted)
    normal = vectors[:, 0]
    return canonical_line(normal, normal @ centre)

def ransac_line(points, threshold=1.4, iterations=2000, min_inliers=35, seed=11):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(iterations):
        sample_ids = rng.choice(len(points), 2, replace=False)
        model = line_from_two(points[sample_ids])
        if model is None:
            continue
        errors = line_error(points, model)
        mask = errors < threshold
        count = int(mask.sum())
        if count < min_inliers:
            continue
        score = (count, -float(np.mean(errors[mask]**2)))
        if best is None or score > best[0]:
            best = (score, model, mask.copy(), sample_ids.copy())
    if best is None:
        raise RuntimeError("No line consensus. Review the threshold/min_inliers.")
    _, sample_model, sample_mask, sample_ids = best
    # Refit to the winning consensus, then reclassify once and refit.
    fitted = tls_line(points[sample_mask])
    mask = line_error(points, fitted) < threshold
    if mask.sum() < min_inliers:
        raise RuntimeError("Line consensus became too small after refitting.")
    fitted = tls_line(points[mask])
    final_mask = line_error(points, fitted) < threshold
    if final_mask.sum() < min_inliers:
        raise RuntimeError("Final line consensus is too small.")
    return {"model": fitted, "mask": final_mask, "sample_model": sample_model,
            "sample_ids": sample_ids, "sample_mask": sample_mask}

line_result = ransac_line(X)
line_model = line_result["model"]
line_mask = line_result["mask"]
print("Best-fit line (a, b, d):", line_model)
print("Normal length:", np.linalg.norm(line_model[:2]))
print("Line inliers:", line_mask.sum())
assert np.isclose(np.linalg.norm(line_model[:2]), 1.0)
assert line_model[2] >= 0

**Parameter choices:** 1.4 is about twice the line's perpendicular noise standard deviation \(1/\sqrt{2}\). At least 35 inliers are required because about 50 points belong to the line. These are sensible starting choices, not universal constants.

With inlier fraction w, sample size s and desired probability p, the usual iteration estimate is \(K\geq\log(1-p)/\log(1-w^s)\). We use 2000 line trials and 3000 circle trials for this small dataset; the sampling work is inexpensive.

### 2.2 Circle samples and radial error
Remove the line consensus. Three noncollinear points determine a sample circle. Reject collinear samples. The error is
\[
\left|\sqrt{(x-c_x)^2+(y-c_y)^2}-r\right|.
\]
We refine the winning circle by minimizing the **sum of squared radial residuals** using `scipy.optimize.minimize`, with a positive radius constraint.

In [ ]:
def circle_from_three(sample):
    p0, p1, p2 = sample
    A = 2*np.vstack((p1-p0, p2-p0))
    if abs(np.linalg.det(A)) < 1e-8:
        return None
    rhs = np.array([p1@p1-p0@p0, p2@p2-p0@p0])
    centre = np.linalg.solve(A, rhs)
    radius = np.linalg.norm(p0-centre)
    model = np.r_[centre, radius]
    return model if np.all(np.isfinite(model)) and radius > 0 else None

def circle_error(points, model):
    return np.abs(np.linalg.norm(points-model[:2], axis=1)-model[2])

def fit_circle(points, start):
    def objective(model):
        residual = np.linalg.norm(points-model[:2], axis=1)-model[2]
        return float(residual @ residual)
    result = minimize(objective, start, method="L-BFGS-B",
                      bounds=[(None, None), (None, None), (1e-6, None)])
    if not result.success:
        raise RuntimeError(f"Circle optimization failed: {result.message}")
    return result.x

def ransac_circle(points, threshold=1.5, iterations=3000, min_inliers=25, seed=22):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(iterations):
        sample_ids = rng.choice(len(points), 3, replace=False)
        model = circle_from_three(points[sample_ids])
        if model is None:
            continue
        errors = circle_error(points, model)
        mask = errors < threshold
        count = int(mask.sum())
        if count < min_inliers:
            continue
        score = (count, -float(np.mean(errors[mask]**2)))
        if best is None or score > best[0]:
            best = (score, model, mask.copy(), sample_ids.copy())
    if best is None:
        raise RuntimeError("No circle consensus. Review the threshold/min_inliers.")
    _, sample_model, sample_mask, sample_ids = best
    fitted = fit_circle(points[sample_mask], sample_model)
    mask = circle_error(points, fitted) < threshold
    if mask.sum() < min_inliers:
        raise RuntimeError("Circle consensus became too small after refitting.")
    fitted = fit_circle(points[mask], fitted)
    final_mask = circle_error(points, fitted) < threshold
    if final_mask.sum() < min_inliers:
        raise RuntimeError("Final circle consensus is too small.")
    return {"model": fitted, "mask": final_mask, "sample_model": sample_model,
            "sample_ids": sample_ids, "sample_mask": sample_mask}

remaining_ids = np.flatnonzero(~line_mask)
remaining = X[remaining_ids]
circle_result = ransac_circle(remaining)
circle_model = circle_result["model"]
circle_mask = np.zeros(len(X), dtype=bool)
circle_mask[remaining_ids[circle_result["mask"]]] = True
print("Remaining points:", len(remaining))
print("Best-fit circle (cx, cy, radius):", circle_model)
print("Circle inliers:", circle_mask.sum())
print("Centre error:", np.linalg.norm(circle_model[:2] - [x0_gt, y0_gt]))
print("Radius error:", abs(circle_model[2] - r_gt))

The circle threshold is 1.5, about 2.4 times the radial noise standard deviation (0.625). At least 25 inliers are required because removing the line also removes some circle points near its intersections. Some model overlap is unavoidable.

### 2.3 Required combined plot
A dashed model comes from the winning random sample. A solid model is refitted using the consensus. The two line sample points and three circle sample points are marked separately. The circle sample indices are relative to `remaining`, so we map them back to the original points.

In [ ]:
def draw_line(ax, model, **kwargs):
    a, b, d = model
    if abs(b) > 1e-10:
        xx = np.array([-16, 16])
        ax.plot(xx, (d-a*xx)/b, **kwargs)
    else:
        ax.axvline(d/a, **kwargs)

fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(X[:, 0], X[:, 1], color="0.65", s=22, label="All points")
ax.scatter(X[line_mask, 0], X[line_mask, 1], color="tab:blue", s=28, label="Line inliers")
ax.scatter(X[circle_mask, 0], X[circle_mask, 1], color="tab:orange", s=28, label="Circle inliers")
line_sample = X[line_result["sample_ids"]]
circle_sample = X[remaining_ids[circle_result["sample_ids"]]]
ax.scatter(line_sample[:, 0], line_sample[:, 1], s=130, marker="s",
           facecolors="none", edgecolors="navy", linewidths=2, label="Line sample (2)")
ax.scatter(circle_sample[:, 0], circle_sample[:, 1], s=160, marker="*",
           color="red", label="Circle sample (3)")
draw_line(ax, line_result["sample_model"], color="cyan", linestyle="--", label="Sample line")
draw_line(ax, line_model, color="navy", linewidth=2, label="Best-fit line")
for model, colour, style, label in [
        (circle_result["sample_model"], "red", "--", "Sample circle"),
        (circle_model, "darkorange", "-", "Best-fit circle")]:
    ax.add_patch(plt.Circle(model[:2], model[2], fill=False, color=colour,
                           linestyle=style, linewidth=2, label=label))
ax.add_patch(plt.Circle((x0_gt, y0_gt), r_gt, fill=False, color="green",
                       linestyle=":", label="True circle"))
draw_line(ax, canonical_line(np.array([1., 1.]), 2.), color="magenta",
          linestyle=":", label="True line")
ax.set(xlim=(-16, 16), ylim=(-14, 17), xlabel="x", ylabel="y", title="Line-first RANSAC")
ax.set_aspect("equal"); ax.grid(alpha=0.25)
ax.legend(loc="upper left", fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(OUT / "q2_ransac_models.png", dpi=200, bbox_inches="tight")
plt.show()
summary = {"line_a_b_d": line_model.tolist(), "line_inliers": int(line_mask.sum()),
           "circle_cx_cy_r": circle_model.tolist(), "circle_inliers": int(circle_mask.sum()),
           "line_threshold": 1.4, "circle_threshold": 1.5,
           "line_iterations": 2000, "circle_iterations": 3000}
(OUT / "q2_parameters.json").write_text(json.dumps(summary, indent=2))

### 2.4 What happens if the circle is fitted first?
A very large circle has a nearly straight arc. Circle RANSAC may therefore mistake line points for a huge circle, especially if unrestricted radii are allowed. It can also find the real circle; failure is **not guaranteed**. Removing the wrong consensus then damages the later line fit.

Run this experiment. Describe what **your run** actually produces and compare its radius with 10. We deliberately do not restrict the radius using the known ground truth.

In [ ]:
circle_first = ransac_circle(X, seed=33)
first_mask = circle_first["mask"]
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(X[:, 0], X[:, 1], color="0.65", s=20, label="All points")
ax.scatter(X[first_mask, 0], X[first_mask, 1], s=25, color="tab:orange", label="Circle-first inliers")
first_model = circle_first["model"]
ax.add_patch(plt.Circle(first_model[:2], first_model[2], fill=False,
                       color="red", label="Circle-first model"))
left = X[~first_mask]
print("Circle-first centre and radius:", first_model)
print("Circle-first inliers:", first_mask.sum(), "; remaining:", len(left))
try:
    after_circle = ransac_line(left, min_inliers=25, seed=44)
    print("Subsequent line:", after_circle["model"], "; inliers:", after_circle["mask"].sum())
    draw_line(ax, after_circle["model"], color="blue", label="Subsequent line")
except RuntimeError as error:
    print("Subsequent line:", error)
ax.set(xlim=(-16, 16), ylim=(-14, 17), title="Circle-first experiment")
ax.set_aspect("equal"); ax.grid(alpha=0.25); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "q2_circle_first.png", dpi=200, bbox_inches="tight")
plt.show()

**For the report:** include the combined plot, chosen thresholds, consensus requirements, iterations, estimated parameters and an explanation of the fitting order. Explain why perpendicular and radial errors are suitable. Include a few important lines of your RANSAC code.

Optional check: rerun with different RANSAC seeds and see whether the fitted parameters are stable. Do not change the generated data when comparing just the RANSAC seed.

**Commit checkpoint:** commit your Q2 work.

## 3. Place an image on a planar surface
You must use **two image pairs of your own choice**. Each pair needs a scene with a reasonably flat surface (a wall, sign or poster area) and an image to place there. You could use your own building photo with a club logo, and a noticeboard photo with an event poster. Explain your own choices in the report.

Create `a2images/my_pairs` and add:
- `scene1.jpg`, `overlay1.png`
- `scene2.jpg`, `overlay2.png`

You can use different extensions by editing the paths below. For practice only, the supplied pair is `wadham.jpg` and `flag.png`; replace the first pair's paths with these if you want to learn the clicking process before using your own pair.

The following code scales scenes to at most 1200 pixels on the longest side for easier clicking. The saved outputs and clicked coordinates use this working resolution.

### 3.1 Load functions and enable interactive figures
The `%matplotlib widget` line enables mouse clicks in the notebook. If it fails, make sure `ipympl` was installed in **the selected kernel**, restart the kernel and run the preceding sections again. You may also need to close and reopen the notebook.

In [ ]:
%matplotlib widget

def resize_max(im, maximum=1200):
    scale = min(1.0, maximum/max(im.shape[:2]))
    if scale < 1:
        return cv.resize(im, None, fx=scale, fy=scale, interpolation=cv.INTER_AREA)
    return im

def load_pair(scene_path, overlay_path):
    scene = resize_max(read_bgr(scene_path))
    overlay = cv.imread(str(overlay_path), cv.IMREAD_UNCHANGED)
    if overlay is None:
        raise FileNotFoundError(f"Could not read {overlay_path}. Add your images or edit the paths.")
    if overlay.ndim == 2:
        overlay = cv.cvtColor(overlay, cv.COLOR_GRAY2BGR)
    return scene, overlay

def click_four(scene, key):
    # Reuse the dictionary if this cell is rerun.
    selections[key] = []
    fig, ax = plt.subplots(figsize=(9, 7))
    show_bgr(scene, "Click TL, TR, BR, BL on the target plane", ax)
    def on_click(event):
        if event.inaxes != ax or event.button != 1 or len(selections[key]) >= 4:
            return
        selections[key].append([float(event.xdata), float(event.ydata)])
        index = len(selections[key])
        ax.plot(event.xdata, event.ydata, "ro")
        ax.text(event.xdata, event.ydata, str(index), color="yellow", fontsize=12)
        fig.canvas.draw_idle()
        if index == 4:
            polygon = np.asarray(selections[key], dtype=np.float32)
            if not cv.isContourConvex(polygon.reshape(-1, 1, 2)) or abs(cv.contourArea(polygon)) < 25:
                print("Invalid quadrilateral. Rerun the clicking cell and select TL, TR, BR, BL.")
                selections[key] = []
            else:
                (OUT / f"q3_{key}_corners.json").write_text(json.dumps(selections[key], indent=2))
                print(f"{key}: four corners saved. Run the next cell.")
            fig.canvas.mpl_disconnect(connection)
    connection = fig.canvas.mpl_connect("button_press_event", on_click)
    return fig, on_click  # keep the callback alive

def overlay_on_plane(scene, overlay, corners, alpha=0.85):
    destination = np.asarray(corners, dtype=np.float32)
    if destination.shape != (4, 2):
        raise ValueError("Click exactly four corners before running this cell.")
    if not cv.isContourConvex(destination.reshape(-1, 1, 2)):
        raise ValueError("Corners must follow TL, TR, BR, BL around a convex surface.")
    oh, ow = overlay.shape[:2]
    source = np.float32([[0, 0], [ow-1, 0], [ow-1, oh-1], [0, oh-1]])
    H = cv.getPerspectiveTransform(source, destination)
    colour = overlay[:, :, :3].astype(np.float32)
    opacity = (overlay[:, :, 3].astype(np.float32)/255 if overlay.shape[2] == 4
               else np.ones((oh, ow), dtype=np.float32))
    height, width = scene.shape[:2]
    # Premultiplied alpha avoids a dark edge during interpolation.
    warped_colour = cv.warpPerspective(colour*opacity[..., None], H, (width, height))
    warped_opacity = cv.warpPerspective(opacity, H, (width, height))
    result = scene.astype(np.float32)*(1-alpha*warped_opacity[..., None]) + alpha*warped_colour
    return np.clip(result, 0, 255).astype(np.uint8), H

selections = {}

### 3.2 First pair: choose the plane
Run the cell, then **click inside its output figure** in this order: top left, top right, bottom right, bottom left of the surface. Turn off toolbar pan/zoom modes before clicking. Wait for the message saying four corners were saved. If you misclick, rerun this cell and click again.

In [ ]:
scene1, overlay1 = load_pair(DATA / "my_pairs" / "scene1.jpg",
                              DATA / "my_pairs" / "overlay1.png")
picker1 = click_four(scene1, "pair1")
plt.show()

### 3.3 Blend and save the first pair
Only run this after selecting four corners. `H` maps overlay-image coordinates into scene-image coordinates. The opacity is 0.85; set it to 1 for an opaque overlay.

In [ ]:
result1, H1 = overlay_on_plane(scene1, overlay1, selections["pair1"], alpha=0.85)
cv.imwrite(str(OUT / "q3_pair1.png"), result1)
np.save(OUT / "q3_pair1_H.npy", H1)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
show_bgr(scene1, "Original", axs[0]); show_bgr(result1, "Planar overlay", axs[1])
fig.tight_layout()
fig.savefig(OUT / "q3_pair1_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Homography for pair 1:\n", H1)

### 3.4 Second pair: select four corners
Use a second scene and overlay of your own choice. Click in the same TL, TR, BR, BL order.

In [ ]:
scene2, overlay2 = load_pair(DATA / "my_pairs" / "scene2.jpg",
                              DATA / "my_pairs" / "overlay2.png")
picker2 = click_four(scene2, "pair2")
plt.show()

### 3.5 Blend and save the second pair

In [ ]:
result2, H2 = overlay_on_plane(scene2, overlay2, selections["pair2"], alpha=0.85)
cv.imwrite(str(OUT / "q3_pair2.png"), result2)
np.save(OUT / "q3_pair2_H.npy", H2)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
show_bgr(scene2, "Original", axs[0]); show_bgr(result2, "Planar overlay", axs[1])
fig.tight_layout()
fig.savefig(OUT / "q3_pair2_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Homography for pair 2:\n", H2)

**For the report:** show both results, identify the four corner correspondences and explain the perspective warp and alpha blend briefly. Explain your choices. A homography is appropriate for the chosen plane; poor corner clicks or a curved surface can make the overlay look wrong.

The corner JSON files save your clicks. To reuse them later, set `selections["pair1"] = json.loads((OUT / "q3_pair1_corners.json").read_text())` after loading `scene1` and `overlay1`, and similarly for pair 2. Avoid rerunning the clicking cell if you want to reuse saved points.

**Commit checkpoint:** commit both pairs and your notebook. Ensure you have permission to reuse your chosen images and record any sources.

## 4. Align each PCB to the template
The template is **AECC1.jpg**. Compare **AAAA1, ADAD1, AECC2, AECC3 and AECC4** against it.

For each pair: SIFT features → BFMatcher → Lowe ratio test → RANSAC homography → warp moving image to template → absolute difference. Unlike Q2, Q4 permits OpenCV's RANSAC.

We resize each image to at most 1600 pixels on the longest side to reduce memory and processing time. The homography and difference images use these working coordinates. Do not compare a resized image directly with a full-size image.

### 4.1 Load the template and detect SIFT features
Switch back to normal static notebook plots. This section may be run after Q2 if you want to postpone your image choices for Q3.

In [ ]:
%matplotlib inline
PCB_MAX_SIDE = 1600

def pcb_resize(im):
    factor = min(1.0, PCB_MAX_SIDE/max(im.shape[:2]))
    return (cv.resize(im, None, fx=factor, fy=factor, interpolation=cv.INTER_AREA)
            if factor < 1 else im)

template = pcb_resize(read_bgr(DATA / "pcbs" / "AECC1.jpg"))
template_gray = cv.cvtColor(template, cv.COLOR_BGR2GRAY)
sift = cv.SIFT_create(nfeatures=6000)
kp_template, des_template = sift.detectAndCompute(template_gray, None)
print("Template working shape:", template.shape)
print("Template SIFT keypoints:", len(kp_template))
keypoint_view = cv.drawKeypoints(template, kp_template, None,
                               flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
show_bgr(keypoint_view, "Template SIFT keypoints")
plt.show()

### 4.2 Match, estimate H, warp and subtract
`BFMatcher(cv.NORM_L2)` is suitable for SIFT descriptors. Keep the first match if its distance is less than 0.75 times the second-best distance. At least four correspondences are required for a homography, but many spatially distributed inliers are preferable.

The matched points go **from moving PCB to template**, so `warpPerspective` aligns the PCB with the template. The RANSAC reprojection threshold is 4 working-image pixels. Only homography inliers are drawn, up to 80 for readability.

Warped pixels outside the moving image have no valid comparison. A warped validity mask excludes them. We slightly erode this mask to remove interpolation edges. Differences are grayscale absolute intensity differences on the 0–255 scale; we do not normalize each image separately.

In [ ]:
def align_pcb(path):
    moving = pcb_resize(read_bgr(path))
    moving_gray = cv.cvtColor(moving, cv.COLOR_BGR2GRAY)
    kp_moving, des_moving = sift.detectAndCompute(moving_gray, None)
    if des_template is None or des_moving is None:
        raise RuntimeError(f"No SIFT descriptors for {path.name}")
    matcher = cv.BFMatcher(cv.NORM_L2)
    pairs = matcher.knnMatch(des_moving, des_template, k=2)
    good = [pair[0] for pair in pairs if len(pair) == 2 and pair[0].distance < 0.75*pair[1].distance]
    if len(good) < 4:
        raise RuntimeError(f"Only {len(good)} ratio-test matches for {path.name}")
    source = np.float32([kp_moving[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    destination = np.float32([kp_template[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    cv.setRNGSeed(0)
    H, inlier_mask = cv.findHomography(source, destination, cv.RANSAC, 4.0)
    if H is None or inlier_mask is None or not np.all(np.isfinite(H)):
        raise RuntimeError(f"Homography failed for {path.name}")
    keep = inlier_mask.ravel().astype(bool)
    if keep.sum() < 4:
        raise RuntimeError(f"Too few homography inliers for {path.name}")
    projected = cv.perspectiveTransform(source[keep], H)
    reprojection_error = np.linalg.norm(projected-destination[keep], axis=2).mean()
    height, width = template.shape[:2]
    aligned = cv.warpPerspective(moving, H, (width, height))
    valid = cv.warpPerspective(np.full(moving.shape[:2], 255, dtype=np.uint8),
                               H, (width, height), flags=cv.INTER_NEAREST)
    valid = cv.erode(valid, np.ones((5, 5), np.uint8)) > 0
    if not valid.any():
        raise RuntimeError(f"No overlapping region for {path.name}")
    difference = cv.absdiff(template_gray, cv.cvtColor(aligned, cv.COLOR_BGR2GRAY))
    difference[~valid] = 0
    inlier_matches = [match for match, flag in zip(good, keep) if flag]
    inlier_matches.sort(key=lambda m: m.distance)
    matches_view = cv.drawMatches(moving, kp_moving, template, kp_template,
                                  inlier_matches[:80], None,
                                  flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    return {"name": path.stem, "moving": moving, "aligned": aligned, "diff": difference,
            "valid": valid, "matches": matches_view, "H": H,
            "good_matches": len(good), "inliers": int(keep.sum()),
            "mean_reprojection_error": float(reprojection_error),
            "mean_abs_difference": float(difference[valid].mean())}

### 4.3 Process all five images
This may take a little time. Every PCB needs a difference image. The table gives match counts and residuals; inspect the visual alignment as well, because match count alone does not prove the warp is correct.

If one image fails, the cell reports it and continues. **Do not leave a failure unresolved for the final report.** Check the image, increase `nfeatures`, or try a ratio threshold of 0.8; then rerun the template cell, alignment function and loop.

In [ ]:
pcb_paths = sorted(p for p in (DATA / "pcbs").glob("*.jpg") if p.name != "AECC1.jpg")
assert len(pcb_paths) == 5, f"Expected 5 comparison images, found {len(pcb_paths)}"
pcb_results = []
pcb_failures = []
for path in pcb_paths:
    print("Processing", path.name, "...", flush=True)
    try:
        result = align_pcb(path)
    except RuntimeError as error:
        pcb_failures.append(path.name)
        print("FAILED:", error)
        continue
    pcb_results.append(result)
    name = result["name"]
    cv.imwrite(str(OUT / f"q4_{name}_matches.jpg"), result["matches"])
    cv.imwrite(str(OUT / f"q4_{name}_aligned.jpg"), result["aligned"])
    cv.imwrite(str(OUT / f"q4_{name}_difference.png"), result["diff"])
    cv.imwrite(str(OUT / f"q4_{name}_valid_mask.png"), result["valid"].astype(np.uint8)*255)
    np.save(OUT / f"q4_{name}_H.npy", result["H"])
    print(f"  Matches={result['good_matches']}, inliers={result['inliers']}, "
          f"reprojection={result['mean_reprojection_error']:.2f}px, "
          f"mean difference={result['mean_abs_difference']:.2f}")

columns = ["name", "good_matches", "inliers", "mean_reprojection_error", "mean_abs_difference"]
with open(OUT / "q4_alignment_summary.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows({key: result[key] for key in columns} for result in pcb_results)
print("Finished. Failed images:", pcb_failures)

### 4.4 Display all difference images and one match example
Use the **same intensity scale** for every difference image. White regions mean larger differences. Masked nonoverlap is black but must not be interpreted as a perfect match; use the saved validity masks if necessary.

In [ ]:
if pcb_results:
    fig, axs = plt.subplots(2, 3, figsize=(13, 8))
    for ax, result in zip(axs.flat, pcb_results):
        ax.imshow(result["diff"], cmap="gray", vmin=0, vmax=255)
        ax.set_title(result["name"])
        ax.axis("off")
    for ax in list(axs.flat)[len(pcb_results):]:
        ax.axis("off")
    fig.suptitle("Absolute differences after alignment (0-255)")
    fig.tight_layout()
    fig.savefig(OUT / "q4_all_differences.png", dpi=220, bbox_inches="tight")
    plt.show()
    show_bgr(pcb_results[0]["matches"], "Example: SIFT homography-inlier matches")
    plt.show()

### 4.5 Inspect an alignment and interpret the differences
Change `selected_index` from 0 to 4 to inspect each result. Bright traces around many flat board edges can indicate an inaccurate homography. Local bright regions may be caused by actual component changes, illumination, reflections or raised parts.

**Why do the USB Type B port, capacitors and reset button show large differences?** A single homography aligns one plane, approximately the PCB surface. Raised components lie outside that plane, so camera viewpoint changes cause parallax that the planar warp cannot remove. Their visible sides and occlusions change too. Shiny metal and component surfaces also change brightness with lighting/viewpoint. Therefore a bright difference is not automatically a defect.

In [ ]:
selected_index = 0
if pcb_results:
    selected = pcb_results[selected_index]
    fig, axs = plt.subplots(1, 3, figsize=(14, 5))
    show_bgr(template, "Template AECC1", axs[0])
    show_bgr(selected["aligned"], selected["name"] + " aligned", axs[1])
    axs[2].imshow(selected["diff"], cmap="gray", vmin=0, vmax=255)
    axs[2].set_title("Absolute difference"); axs[2].axis("off")
    fig.tight_layout()
    fig.savefig(OUT / f"q4_{selected['name']}_comparison.png", dpi=180, bbox_inches="tight")
    plt.show()

**For the report:** include an example of the SIFT matches, all five difference images, the ratio/reprojection thresholds, working resolution, and the explanation of raised components. Compare good and poor alignment regions. Do not label every bright patch as a defect.

**Commit checkpoint:** commit your Q4 work.

## 5. GitHub: record your progress
The assignment asks for regular commits over a reasonable duration. Begin now and commit after meaningful work; do not wait until everything is finished.

1. In the project folder, create a file called `.gitignore` with:

```gitignore
.venv/
__pycache__/
.ipynb_checkpoints/
```

2. In VS Code, open **Source Control** (Ctrl+Shift+G) and choose **Initialize Repository**. If Git is missing, install Git for Windows and restart VS Code.
3. Save the notebook with Ctrl+S. In Source Control, stage relevant files using **+**, type a message such as `Set up assignment notebook and data`, then click **Commit**.
4. Use **Publish Branch / Publish to GitHub**, sign in and choose a repository name. For grader access, make it public if your course policy permits; otherwise arrange the required access. Do not commit your `.venv` folder.
5. After Q1, Q2, Q3 and Q4, save, stage, commit and **Sync Changes** again. Use messages describing what you actually changed.
6. Add your **GitHub profile URL** to the report, as requested. Adding the assignment repository URL too helps locate the work.

If you already have a repository, open its project folder and use its existing Source Control setup rather than initializing another one.

## 6. Prepare the report — maximum four pages
Use this as a planning guide, not a strict page template:

| Page | Suggested content |
|---|---|
| 1 | Compact title, name/index/GitHub; Q1 method, overlay and largest-circle parameters |
| 2 | Q2 essential code, combined plot, numerical results, thresholds and circle-first discussion |
| 3 | Q3 two image pairs/results, four-point mapping and blending explanation |
| 4 | Q4 example matches, five difference images, parameters and interpretation |

Do not add a separate cover page. Make figures readable, label them and refer to them in the text. Include selected important code rather than the whole notebook. State limitations and observations in your own words. The assignment says grading uses the PDF, so include all essential evidence in it.

Before submission:
- All four questions and their subparts are addressed.
- Q1 lists the largest-circle parameters and sigma range, with coordinate resolution stated.
- Q2 uses your own RANSAC and shows sample models, best-fit models, inliers and sample points.
- Q3 contains two image pairs you chose, with sources/permission where relevant.
- Q4 contains differences for all five comparison PCBs and explains raised components.
- Your name, index number and GitHub profile link appear in the PDF.
- The PDF has no more than four pages and is named `your_index_a02.pdf`.

## Documentation
- [VS Code notebooks](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)
- [VS Code notebook kernels](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management)
- [SciPy Gaussian Laplacian](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_laplace.html)
- [SciPy maximum filter](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.maximum_filter.html)
- [SciPy minimize](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)
- [OpenCV feature matching and homography](https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html)
- [Boyd and Vandenberghe, section 1.1](https://web.stanford.edu/~boyd/cvxbook/) — read this as requested in Q2.